In [3]:
from __future__ import annotations

from typing import Optional, Union
import math
import warnings
import os

import numpy as np
import pandas as pd


K_B_EV_PER_K = 8.617333262145e-5
DEFAULT_T_K = 300.0
DEFAULT_D_CUT_ANG = 1.66 

UV_FROM_GAP_B = 3.05
UV_FROM_GAP_BETA = 0.736

DEFAULT_OMEGA_IR_eV = 0.02

FIXED_INPUT_BASENAME = "Hamaker-Layered TMHs-input.xlsx"
FIXED_OUTPUT_BASENAME = "Hamaker-Layered TMHs-output.xlsx"


LI3_TERMS = 60



def omega_uv_from_gap(Eg_eV: Optional[float]) -> Optional[float]:
    
    if Eg_eV is None:
        return None
    try:
        Eg = float(Eg_eV)
    except Exception:
        return None
    if not np.isfinite(Eg) or Eg <= 0:
        return None
    return UV_FROM_GAP_B * (Eg ** UV_FROM_GAP_BETA)


def resolve_fixed_input_path(input_path: Optional[str] = None) -> str:
    if input_path is not None:
        return input_path
    fixed_path = os.path.join(os.getcwd(), FIXED_INPUT_BASENAME)
    if not os.path.exists(fixed_path):
        raise FileNotFoundError(
            f"Required input file not found: {fixed_path}\n"
            f"Please place '{FIXED_INPUT_BASENAME}' in the working directory."
        )
    return fixed_path


def _as_float_or_none(v) -> Optional[float]:
    if v is None:
        return None
    if pd.isna(v):
        return None
    try:
        x = float(v)
    except Exception:
        return None
    if not np.isfinite(x):
        return None
    return x


def _require_positive_gap(Eg_raw, material: str = "UNKNOWN") -> float:
    Eg = _as_float_or_none(Eg_raw)
    if Eg is None or Eg <= 0:
        raise ValueError(
            f"Metal/zero-gap detected: Bandgap (ev)={Eg_raw}. "
            f"This Hamaker model requires Eg > 0. "
            f"For metals you need a Drude model (ωp, γ) or exclude the row. "
            f"Material={material}"
        )
    return float(Eg)



_INV_K3 = 1.0 / (np.arange(1, LI3_TERMS + 1, dtype=float) ** 3)


def li3_fast(z: np.ndarray) -> np.ndarray:
    
    z = np.asarray(z, dtype=float)
    z = np.clip(z, 0.0, 1.0 - 1e-12)
    out = np.zeros_like(z)

    zk = z.copy()
    for k in range(LI3_TERMS):
        out += zk * _INV_K3[k]
        zk *= z
    return out



def build_axis_eps_i_xi_fn(
    eps_inf: float,
    eps_0: Optional[float],
    alpha: float,
    Eg_eV: float,  
    omega_uv_eV: Optional[float] = None,
    omega_ir_eV: float = DEFAULT_OMEGA_IR_eV,
):
   
    if eps_inf is None or not np.isfinite(float(eps_inf)):
        raise ValueError("eps_inf is required and must be finite.")
    eps_inf = float(eps_inf)

    if alpha is None or (not np.isfinite(float(alpha))) or float(alpha) <= 0:
        alpha = 1.0
    alpha = float(alpha)

    C = eps_inf - 1.0

    w_uv = omega_uv_eV
    if w_uv is None:
        w_uv = omega_uv_from_gap(Eg_eV)
    if w_uv is None or (not np.isfinite(float(w_uv))) or float(w_uv) <= 0:
        w_uv = 3.0
    w_uv = float(w_uv)

    omega_ir_eV = float(omega_ir_eV) if omega_ir_eV is not None else DEFAULT_OMEGA_IR_eV
    if (not np.isfinite(omega_ir_eV)) or omega_ir_eV <= 0:
        omega_ir_eV = DEFAULT_OMEGA_IR_eV

    S_IR = None
    if eps_0 is not None:
        try:
            S = float(eps_0) - eps_inf
        except Exception:
            S = None
        if S is not None and np.isfinite(S) and S > 0:
            S_IR = float(S)

    def eps_i_xi(xi: float) -> float:
        xi = float(xi)
        elec = 1.0 + (C / (1.0 + (xi / w_uv) ** alpha))
        ir = (S_IR / (1.0 + (xi / omega_ir_eV) ** 2)) if S_IR is not None else 0.0
        val = elec + ir
        if not np.isfinite(val) or val <= 0:
            return 1.0
        return float(val)

    return eps_i_xi



def hamaker_from_row_anisotropic(
    row: pd.Series,
    *,
    Emax_eV: Optional[float] = 300.0,
    n_max: int = 1000,
    tol: float = 0.0,
    T_K: float = DEFAULT_T_K,
    Npsi: int = 360,        
    verbose: bool = False,
) -> float:
    

    material = row.get("Material", row.get("Formula", "UNKNOWN"))

    
    eps_inf_x = float(row["ε_∞ (11)"])
    eps_inf_y = float(row["ε_∞ (22)"])
    eps_inf_z = float(row["ε_∞ (33)"])

   
    eps_0_x = _as_float_or_none(row.get("ε_0 (11)", None))
    eps_0_y = _as_float_or_none(row.get("ε_0 (22)", None))
    eps_0_z = _as_float_or_none(row.get("ε_0 (33)", None))

    
    Eg_eV = _require_positive_gap(row.get("Bandgap (ev)", None), material=material)

    alpha = _as_float_or_none(row.get("alpha", None))
    if alpha is None or alpha <= 0:
        alpha = 1.0
    alpha = float(alpha)

    omega_ir_eV = _as_float_or_none(row.get("omega_ir_eV", None))
    if omega_ir_eV is None:
        omega_ir_eV = DEFAULT_OMEGA_IR_eV

    
    w_uv_base = omega_uv_from_gap(Eg_eV) or 3.0

    
    Cx, Cy, Cz = eps_inf_x - 1.0, eps_inf_y - 1.0, eps_inf_z - 1.0
    C_in_plane = 0.5 * (Cx + Cy)

    w_uv_x = w_uv_base
    w_uv_y = w_uv_base
    if C_in_plane > 0 and Cz > 0:
        w_uv_z = w_uv_base * ((C_in_plane / Cz) ** (1.0 / alpha))
        if (not np.isfinite(w_uv_z)) or w_uv_z <= 0:
            w_uv_z = w_uv_base
    else:
        w_uv_z = w_uv_base

    eps_x_fn = build_axis_eps_i_xi_fn(eps_inf_x, eps_0_x, alpha, Eg_eV, omega_uv_eV=w_uv_x, omega_ir_eV=omega_ir_eV)
    eps_y_fn = build_axis_eps_i_xi_fn(eps_inf_y, eps_0_y, alpha, Eg_eV, omega_uv_eV=w_uv_y, omega_ir_eV=omega_ir_eV)
    eps_z_fn = build_axis_eps_i_xi_fn(eps_inf_z, eps_0_z, alpha, Eg_eV, omega_uv_eV=w_uv_z, omega_ir_eV=omega_ir_eV)

    kT = K_B_EV_PER_K * float(T_K)
    delta_E = 2.0 * math.pi * kT

    
    if Emax_eV is not None:
        Emax = float(Emax_eV)
        if Emax > 0 and np.isfinite(Emax):
            n_max_eff = int(math.floor(Emax / delta_E))
            n_max_eff = max(n_max_eff, 0)
        else:
            n_max_eff = int(n_max)
    else:
        n_max_eff = int(n_max)

    
    Npsi = int(Npsi)
    if Npsi < 60:
        Npsi = 60
    psis = np.linspace(0.0, 2.0 * math.pi, Npsi + 1)
    cos2 = np.cos(psis) ** 2
    sin2 = 1.0 - cos2

    def psi_integral_Li3_fast(xi: float) -> float:
        ex = float(eps_x_fn(xi))
        ey = float(eps_y_fn(xi))
        ez = float(eps_z_fn(xi))

        
        A2 = ez * (ex * cos2 + ey * sin2)
        A2 = np.maximum(A2, 0.0)
        A = np.sqrt(A2)

        
        denom = A + 1.0
        r = np.where(denom > 0, (A - 1.0) / denom, 0.0)
        r = np.clip(r, -1.0 + 1e-12, 1.0 - 1e-12)

        r2 = r * r
        vals = li3_fast(r2)
        return float(np.trapz(vals, psis))

    
    I0 = psi_integral_Li3_fast(0.0)
    s = 0.5 * I0

    
    for n in range(1, n_max_eff + 1):
        xi_n = delta_E * n
        In = psi_integral_Li3_fast(xi_n)
        s += In
        if n > 50 and tol and tol > 0 and abs(In) < tol:
            break

    if verbose:
        print(f"[dbg] {material}: delta_E={delta_E:.6f} eV, n_max={n_max_eff}, Npsi={Npsi}, LI3_TERMS={LI3_TERMS}")

   
    H_eV = (3.0 * kT / (4.0 * math.pi)) * s
    return float(H_eV)


def energies_from_H(H_eV: float, f_pauling: float, d_cut_A: float = DEFAULT_D_CUT_ANG):
    E_vdW_meV_A2 = (float(H_eV) / (12.0 * math.pi * float(d_cut_A) ** 2)) * 1000.0
    f = 0.0
    if f_pauling is not None and np.isfinite(float(f_pauling)):
        f = float(f_pauling)
        if f < 0 or f >= 1:
            f = 0.0
    return float(E_vdW_meV_A2), float(E_vdW_meV_A2 / (1.0 - f))



REQUIRED = [
    "Material", "Density (g cm-3)", "Bandgap (ev)", "Ionicity", "alpha",
    "ε_∞ (11)", "ε_∞ (22)", "ε_∞ (33)", "ε_0 (11)", "ε_0 (22)", "ε_0 (33)"
]


def run_from_excel(
    input_path: Optional[str] = None,
    output_path: str = FIXED_OUTPUT_BASENAME,
    sheet_name: Union[int, str] = 0,
    Emax_eV: Optional[float] = 300.0,
    n_max: int = 1000,
    tol: float = 0.0,
    T_K: float = DEFAULT_T_K,
    Npsi: int = 360,       
    verbose: bool = False,
) -> pd.DataFrame:

    input_path = resolve_fixed_input_path(input_path)
    print(f"[i] Using input: {input_path}")

    df = pd.read_excel(input_path, sheet_name=sheet_name)

    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    results = []
    for _, row in df.iterrows():
        material = row.get("Material", row.get("Formula", "UNKNOWN"))
        try:
            H = hamaker_from_row_anisotropic(
                row,
                Emax_eV=Emax_eV,
                n_max=n_max,
                tol=tol,
                T_K=T_K,
                Npsi=Npsi,
                verbose=verbose,
            )
            EvdW, Etotal = energies_from_H(H, row["Ionicity"], DEFAULT_D_CUT_ANG)
            results.append({
                "H_eV": H,
                "E_vdW_meV_A2": EvdW,
                "E_total_meV_A2": Etotal,
                "status": "ok",
                "error": ""
            })
        except Exception as e:
            warnings.warn(f"Failed on {material}: {e}")
            results.append({
                "H_eV": np.nan,
                "E_vdW_meV_A2": np.nan,
                "E_total_meV_A2": np.nan,
                "status": "failed",
                "error": str(e)
            })

    final_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
    final_df.to_excel(output_path, index=False)
    print(f"[✓] Results written to: {os.path.abspath(output_path)}")
    return final_df



if __name__ == "__main__":
    final = run_from_excel(
        Emax_eV=300.0,   
        n_max=1000,       
        tol=0.0,          
        T_K=DEFAULT_T_K,
        Npsi=360,        
        verbose=False,
    )
    print(final.head())


[i] Using input: C:\Users\USER\4-Hamaker-Ratio\8-Dielectric&BE-Layered TMHs\2-Binding Energy\Hamaker-Layered TMH-Pollini-input.xlsx
[✓] Results written to: C:\Users\USER\4-Hamaker-Ratio\8-Dielectric&BE-Layered TMHs\2-Binding Energy\Hamaker-Layered TMH-Pollini-output.xlsx
  Material  Density (g cm-3)  Bandgap (ev)  Ionicity  ε_∞ (11)  ε_∞ (22)  \
0    CdCl2              4.08          5.90    0.4174  5.615634  5.615634   
1    CdBr2              5.23          4.60    0.3318  3.500000  3.500000   
2     CdI2              5.71          3.32    0.2096  6.547530  6.547530   
3    CoCl2              3.40          4.69    0.3361  5.197220  5.197220   
4    CoBr2              5.05          3.37    0.2529  5.473827  5.473827   

   ε_∞ (33)  ε_0 (11)  ε_0 (22)  ε_0 (33)     alpha      H_eV  E_vdW_meV_A2  \
0  5.615634       NaN       NaN       NaN  1.760908  2.116628     20.374980   
1  3.500000       NaN       NaN       NaN  1.588718  0.938214      9.031394   
2  6.547530       NaN       NaN   